# ASL Pose Estimation — Colab 訓練 notebook

這個 notebook 對應本地的 `pose_recognition/` 流水線，但跑在 Colab 上比較快、不用煩惱 DirectML/CUDA 環境。

## 兩種使用路線

**路線 A（推薦）**：本地先用 `extract_landmarks.py` 抽好 landmarks → 上傳 CSV → Colab 只負責訓練。
- 上傳資料超小（< 10 MB），訓練幾十秒
- 不用在 Colab 處理 dataset zip

**路線 B**：把整個 dataset zip 上傳 → Colab 抽 landmarks → 訓練。
- 上傳比較大（看你 dataset 大小，可能 100-500 MB）
- 適合「本地連 mediapipe 都不想裝」的情境

依照你的情況選一邊跑就好，不需要兩邊都跑。

---

## 環境設定（**先跑這個**）

Runtime → Change runtime type → 選 **CPU** 或 **T4 GPU** 都可以（MLP 太小用 CPU 也夠快；只有走路線 B 抽 landmarks 時 GPU 加速差異小，因為 MediaPipe Python 預設不吃 GPU）。

In [ ]:
!pip install -q mediapipe==0.10.14 onnx onnxruntime scikit-learn
import torch, mediapipe, sklearn, onnx, onnxruntime
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
print('mediapipe', mediapipe.__version__)
print('onnx', onnx.__version__)
import os
os.makedirs('/content/work/data', exist_ok=True)
os.makedirs('/content/work/models', exist_ok=True)
os.makedirs('/content/work/runs', exist_ok=True)

---

## 路線 A：上傳已抽好的 landmark CSV（推薦）

**前提**：你已經在本地跑過 `python pose_recognition\scripts\extract_landmarks.py --all`，產出三個 CSV：
- `pose_recognition/data/landmarks_train.csv`
- `pose_recognition/data/landmarks_valid.csv`
- `pose_recognition/data/landmarks_test.csv`

執行下一格會跳出檔案選擇器，**一次選三個 CSV 上傳**（按 Ctrl 多選）。

In [ ]:
from google.colab import files
uploaded = files.upload()
import shutil
for name in uploaded:
    if name.startswith('landmarks_') and name.endswith('.csv'):
        shutil.move(name, f'/content/work/data/{name}')
        print(f'  ✓ {name} -> /content/work/data/')
!ls -la /content/work/data/

**走路線 A 的話**：跳過下面「路線 B」的兩個 cell，直接到「訓練 MLP」。

---

## 路線 B：上傳 dataset zip，在 Colab 抽取

在本地用檔案總管壓縮整個 `asl-alphabet` 資料夾成 zip（包含 data.yaml、train/、valid/、test/），檔案會是幾百 MB。

**走路線 A 的話跳過這兩個 cell，直接到下面「訓練 MLP」。**

In [ ]:
# 上傳 dataset zip
from google.colab import files
import os, zipfile
os.makedirs('/content/dataset_root', exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            z.extractall('/content/dataset_root')
        print(f'  ✓ 解壓 {name}')
!ls /content/dataset_root
# 如果解出來是 asl-alphabet/，就讓 DATASET_DIR 指到它

In [ ]:
# 在 Colab 上跑 landmark 抽取（走路線 B 時用）
import os, csv, sys, glob
from pathlib import Path
import cv2, numpy as np
import mediapipe as mp

# 自動找 dataset 路徑
candidates = glob.glob('/content/dataset_root/**/data.yaml', recursive=True)
DATASET_DIR = Path(candidates[0]).parent if candidates else Path('/content/dataset_root/asl-alphabet')
OUTPUT_DIR = Path('/content/work/data')
print('Dataset:', DATASET_DIR)

EXCLUDED = {'J', 'Z'}

def filename_to_letter(name):
    base = name.split('_')[0]
    if base.startswith('nothing'): return 'nothing'
    if base and base[0].isupper(): return base[0]
    return None

def parse_label(p):
    if not p.exists(): return None
    line = p.read_text().strip().split('\n')[0]
    parts = line.split()
    if len(parts) < 5: return None
    return int(parts[0]), tuple(map(float, parts[1:5]))

def crop_bbox(img, bbox, pad=0.2):
    H, W = img.shape[:2]
    cx, cy, w, h = bbox
    x1 = max(0, int((cx - w/2 - pad*w) * W))
    y1 = max(0, int((cy - h/2 - pad*h) * H))
    x2 = min(W, int((cx + w/2 + pad*w) * W))
    y2 = min(H, int((cy + h/2 + pad*h) * H))
    return img[y1:y2, x1:x2] if x2 > x1 and y2 > y1 else img

def normalize(lms):
    pts = np.array([(l.x, l.y, l.z) for l in lms], dtype=np.float64)
    pts -= pts[0]
    s = np.linalg.norm(pts[9])
    if s < 1e-6: return None
    return (pts / s).flatten().tolist()

header = ['label'] + [f'{a}{i}' for i in range(21) for a in ('x','y','z')]

for split in ['train', 'valid', 'test']:
    imgs = sorted((DATASET_DIR / split / 'images').glob('*.jpg'))
    if not imgs: continue
    out = OUTPUT_DIR / f'landmarks_{split}.csv'
    detector = mp.solutions.hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.3)
    ok = no_hand = excluded = 0
    with out.open('w', newline='') as f:
        w = csv.writer(f); w.writerow(header)
        for i, p in enumerate(imgs, 1):
            letter = filename_to_letter(p.name)
            if not letter: continue
            if letter in EXCLUDED: excluded += 1; continue
            img = cv2.imread(str(p))
            lbl = parse_label(DATASET_DIR / split / 'labels' / (p.stem + '.txt'))
            crop = crop_bbox(img, lbl[1], 0.2) if lbl else img
            r = detector.process(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            if not r.multi_hand_landmarks:
                r = detector.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            if not r.multi_hand_landmarks: no_hand += 1; continue
            feat = normalize(r.multi_hand_landmarks[0].landmark)
            if feat: w.writerow([letter] + feat); ok += 1
            if i % 500 == 0: print(f'  {split}: {i}/{len(imgs)} ok={ok}')
    detector.close()
    rate = ok / max(1, len(imgs) - excluded) * 100
    print(f'{split}: total={len(imgs)} ok={ok} excluded={excluded} no_hand={no_hand} 偵測率={rate:.1f}%')

---

## 訓練 MLP（兩條路線匯流到這裡）

In [ ]:
import json, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset

DATA = Path('/content/work/data')
MODELS = Path('/content/work/models'); MODELS.mkdir(exist_ok=True)
RUNS = Path('/content/work/runs'); RUNS.mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

def load(p):
    df = pd.read_csv(p)
    return df.drop(columns=['label']).values.astype(np.float32), df['label'].values

X_tr, y_tr = load(DATA/'landmarks_train.csv')
X_va, y_va = load(DATA/'landmarks_valid.csv')
X_te, y_te = load(DATA/'landmarks_test.csv')

le = LabelEncoder()
y_tr_e = le.fit_transform(y_tr)
y_va_e = le.transform(y_va)
y_te_e = le.transform(y_te)
print(f'類別 ({len(le.classes_)}): {list(le.classes_)}')
print(f'train={len(X_tr)}  valid={len(X_va)}  test={len(X_te)}')

class SignMLP(nn.Module):
    def __init__(self, n_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(63, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, n_classes))
    def forward(self, x): return self.net(x)

def loader(X, y, bs, shuf):
    return DataLoader(TensorDataset(torch.from_numpy(X), torch.from_numpy(y).long()), batch_size=bs, shuffle=shuf)

BATCH = 128; EPOCHS = 100; LR = 1e-3; PATIENCE = 12
tl = loader(X_tr, y_tr_e, BATCH, True)
vl = loader(X_va, y_va_e, BATCH, False)
el = loader(X_te, y_te_e, BATCH, False)

model = SignMLP(len(le.classes_)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=5)

@torch.no_grad()
def evaluate(loader):
    model.eval(); n=c=0; ls=0.0; P=[]; T=[]
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        lg = model(x)
        ls += F.cross_entropy(lg, y, reduction='sum').item()
        p = lg.argmax(1)
        c += (p==y).sum().item(); n += y.size(0)
        P.append(p.cpu().numpy()); T.append(y.cpu().numpy())
    return ls/n, c/n, np.concatenate(T), np.concatenate(P)

best = 0; bad = 0; history = []
for ep in range(1, EPOCHS+1):
    model.train(); rL=rC=rN=0
    for x, y in tl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        lg = model(x)
        loss = F.cross_entropy(lg, y); loss.backward(); opt.step()
        rL += loss.item()*y.size(0); rC += (lg.argmax(1)==y).sum().item(); rN += y.size(0)
    tL, tA = rL/rN, rC/rN
    vL, vA, _, _ = evaluate(vl)
    sched.step(vA)
    history.append({'epoch':ep,'train_loss':tL,'train_acc':tA,'val_loss':vL,'val_acc':vA})
    print(f'ep {ep:3d}  train loss {tL:.4f} acc {tA*100:5.2f}%  |  val loss {vL:.4f} acc {vA*100:5.2f}%')
    if vA > best:
        best = vA; bad = 0
        torch.save({'model_state': model.state_dict(), 'classes': list(le.classes_),
                    'in_dim': 63, 'hidden': (128,64), 'dropout': 0.3}, MODELS/'best_mlp.pt')
    else:
        bad += 1
        if bad >= PATIENCE: print(f'early stop @ ep {ep} (best val acc {best*100:.2f}%)'); break

ckpt = torch.load(MODELS/'best_mlp.pt', map_location=device)
model.load_state_dict(ckpt['model_state'])
_, tstA, yt, yp = evaluate(el)
print(f'\n=== Test acc: {tstA*100:.2f}% ===\n')
rep = classification_report(yt, yp, target_names=le.classes_, digits=3, zero_division=0)
print(rep)
(RUNS/'history.json').write_text(json.dumps(history, indent=2))
(RUNS/'classification_report.txt').write_text(rep)
np.savetxt(RUNS/'confusion_matrix.csv', confusion_matrix(yt, yp), fmt='%d', delimiter=',', header=','.join(le.classes_), comments='')

## 匯出 ONNX

In [ ]:
model.eval()
dummy = torch.randn(1, 63).to(device)
onnx_path = MODELS / 'sign_mlp.onnx'
torch.onnx.export(model, dummy, str(onnx_path),
                  input_names=['landmarks'], output_names=['logits'],
                  dynamic_axes={'landmarks':{0:'batch'}, 'logits':{0:'batch'}},
                  opset_version=13, do_constant_folding=True)
print(f'✓ {onnx_path}')
(MODELS/'classes.json').write_text(json.dumps({'classes': list(le.classes_), 'in_dim': 63}, ensure_ascii=False, indent=2))
print(f'✓ {MODELS}/classes.json')

# 驗證 ONNX
import onnx, onnxruntime as ort
onnx.checker.check_model(onnx.load(str(onnx_path)))
sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
out = sess.run(None, {'landmarks': np.random.randn(1, 63).astype(np.float32)})[0]
print(f'✓ ONNX 推論驗證通過，shape={out.shape}')

## 下載產出檔案到本地

把 `sign_mlp.onnx`、`classes.json` 和 `best_mlp.pt` 一起打包下載。

**下載完，把它們放到本地的：**
- `ai-backend/pose_recognition/models/best_mlp.pt`
- `ai-backend/pose_recognition/models/sign_mlp.onnx`
- `ai-backend/pose_recognition/models/classes.json`
- `frontend/public/models/sign_mlp.onnx`
- `frontend/public/models/classes.json`

In [ ]:
import shutil
shutil.make_archive('/content/pose_outputs', 'zip', '/content/work')
from google.colab import files
files.download('/content/pose_outputs.zip')